# 2.2 Running the model in GreenLight

## Live code in this page

This page uses **live code**. Click the rocket button (🚀) at the top of this page, above the title, then choose "▶ Live Code". The loading may take some time so it is good to start the Live Code before reading on. The Live Code engine is ready when you see a text under the page's title, stating something like "**Launching from mybinder.org: `ready`**.

**Note:** Code that you edit in the live code blocks is not saved. Any changes you make to the code blocks is lost once you move on from the page. Therefore, if you want to keep your work, save a copy of it in a local document.

The code block below defines some imports and functions that will be helpful in the current page. It is run automatically when you run the "▶ Live Code" button. You may have a look at it if you like, but that is not necessary for this course.

In [ ]:
"""
Create a function for displaying dicts in an aesthetic way.
Consider the option of a nested dict (like a GreenLight model definition)
or a "flat" dict (just keys and values)
"""

import pandas as pd

def display_dict(d, dict_type="nested"):
    """
    Below, some dictionaries will be defined in the standard
    GreenLight format for describing models.
    This functions displays such dictionaries in a nicely
    readable format
    """
    if dict_type == "nested":
        df = pd.DataFrame.from_dict(d, orient="index")
        df.index.name = "Variable"
    elif dict_type == "flat":
        df = pd.DataFrame.from_dict(d, orient="index", columns=[""])
    display(df)

## What we have so far

At the end of the previous section, we've finished defining the simple crop model in a Python dict. Run the code block below to load this dict into the variable `crop_definition`.

In [ ]:
"""
Solution to Exercise 6: complete the crop definition dict with description, unit, type, definition, and reference
"""

crop_definition = {
    "W": {
        "description": "Crop dry mass",
        "unit": "g/m2",
        "type": "state",
        "definition": "LUE * ( 1 - exp(-k*LAI) ) * (1/1000000 * Iw)",  # The expression for dW/dT (for states, the definition is the right hand sand of the ODE)
        "init": "75",
        "reference": "Stanghellini (2019), Eq. 2.25. I (MJ/m2/day) is replaced by Iw (W/m2). See also Figure 2.13 there: 75 g/m2 dry mass is a reasonable value for a small crop"
    },
    "LUE": {
        "description": "Light use efficiency",
        "unit": "g crop weight per MJ PAR",
        "type": "const",
        "definition": "4.3",
        "reference": "Stanghellini (2019), Eq. 2.25. See also pg. 49 there: 4.3 is a typical middle value"
    },
    "k": {
        "description": "Light extinction coefficient",
        "unit": "-",
        "type": "const",
        "definition": "0.7",
        "reference": "Stanghellini (2019), Eq. 2.25. See also pg. 39 there: 'For many greenhouse crops, k has a value between 0.6 and 0.8'"
    },
    "LAI": {
        "description": "Leaf area index",
        "unit": "m2 leaf per m2 floor",
        "type": "aux",
        "definition": "W * F_leaves * SLA / 10000",
        "reference": "Stanghellini (2019), Eq. 2.26. "
    },
    "Iw": {
        "description": "Solar PAR radiation",
        "unit": "W PAR per m2",
        "type": "input",
        "definition": "50",
        "reference": "Stanghellini (2019), Eq. 2.25. I (MJ/m2/day) has been replaced by Iw (W/m2). The value of 50 W/m2 is given as an example"
    },
    "F_leaves": {
        "description": "Fraction of crop dry mass partitioned to the leaves",
        "unit": "-",
        "type": "const",
        "definition": "0.8",
        "reference": "Stanghellini (2019), Eq. 2.26. 0.8 is chosen to represent a vegetative crop with mostly leaves"
    },
    "SLA": {
        "description": "Specific leaf area",
        "unit": "cm2/g",
        "type": "const",
        "definition": "250",
        "reference": "Stanghellini (2019), Eq. 2.26. 250 is a middle value for a tomato crop (see Table 2.1, pg. 50 in Stanghellini 2019)"
    }   
 }

# Display the above dict
display_dict(crop_definition)

## Running the model in GreenLight

The code block below loads the crop model definition `dict` that we've created, `crop_definition`, onto a new GreenLight object. This is done by using a Python class constructor - a method that creates a new instance of a class, in this case, an instance of the class `GreenLight`.

In this code, we give `crop_definition` as the `input_prompt` argument for the `GreenLight` constructor. This sets our model definition in the new `GreenLight` instance, called `crop_mdl`.

Then, the method `run()` is called for `crop_mdl`, and the model runs.

In [ ]:
"""
Load the crop model definition to a GreenLight object, and then run it
"""

# Import the GreenLight package
import greenlight

# Create a new GreenLight object. Provide the dict crop_definition as input prompt
crop_mdl = greenlight.GreenLight(input_prompt=crop_definition)

# Run the model
crop_mdl.run()

If all went well, the model run printed out some text, which mentioned that some files were saved once the run was complete. At this moment we don't need to use these files, since the model result is also stored in the object `crop_mdl`.

## Viewing the model results

The code blocks below show some of the information that is stored in `crop_mdl`.

In [ ]:
"""
Display some of the information that is stored in crop_mdl
"""

print("\ncrop_mdl.variables:")
display_dict(crop_mdl.variables, "flat")

print("\ncrop_mdl.var_descriptions:")
display_dict(crop_mdl.var_descriptions, "flat")

print("\ncrop_mdl.var_units:")
display_dict(crop_mdl.var_units, "flat")

print("\ncrop_mdl.var_refs:")
display_dict(crop_mdl.var_refs, "flat")

In [ ]:
"""
Display the model result in a table
"""

print("\ncrop_mdl.full_sol")
display(crop_mdl.full_sol)

## Displaying the model results in a graph

Now that we have seen that some model results have been generated, it would be nice to show them in graphs. The following code block creates a graph for each variable that is not a constant.

In [ ]:
"""
Generate a graph for each variable of the model that is not a constant
"""
# Import matplotlib
import matplotlib.pyplot as plt

for var in crop_mdl.variables.keys():               # Go through all model variables
    if var not in crop_mdl.consts.keys():           # Only plot those that do not have the type "const"
        plt.figure()                                # Create a new figure
        plt.plot(
            crop_mdl.full_sol["Time"],              # Plot the simulated time
            crop_mdl.full_sol[var]                  # Against the variable
        )
        plt.title(crop_mdl.var_descriptions[var])   # Use the variable description as title
        plt.ylabel(                                 # On the y-axis:
            var                                     # Set the variable name
            + " (" + crop_mdl.var_units[var] + ")") # With the units in parentheses
        plt.xlabel("Time (s)")                      # On the x-axis set "Time (s)"
        plt.grid()                                  # Add grid
        plt.show()                                  # Show the plot

## Exercise 7: adjusting the model

```{exercise}
:label: exercise-modify-model
:enumerated: true

Go back to [the code block defining the model](#what-we-have-so-far), modify the model, and generate graphs to show how the model behaviour changes. You are free to choose how to modify the model, but you may consider the following changes:
- Revisit Figure 2.13 from the textbook and consider whether an initial value of 75 g m<sup>-2</sup> dry weight is a good choice
- Check pg. 49 in the textbook and consider other values for $LUE$
- Check pg. 39 of the textbook and consider other values for $k$
- Continue the above examination for the rest of the model variables
```